# Step 4 — Vector Space Model dan Pembobotan TF-IDF

**Tujuan praktikum**
1. Mahasiswa memahami pembobotan *term frequency* (TF), *inverse document frequency* (IDF), dan TF-IDF.
2. Mahasiswa mampu merepresentasikan dokumen dan query sebagai vektor.
3. Mahasiswa mampu menghitung *cosine similarity* dan menghasilkan daftar dokumen terurut (*ranked retrieval*).

**Rumus yang digunakan**

$$w_{t,d} = (1 + \log_{10} tf_{t,d}) \times \log_{10}\frac{N}{df_t}$$

$$\cos(\vec q, \vec d) = \frac{\vec q \cdot \vec d}{\lVert \vec q \rVert \, \lVert \vec d \rVert}$$

In [ ]:
# Jalankan sekali jika pustaka belum terpasang:
# !pip install pandas numpy matplotlib Sastrawi
import pandas as pd
pd.set_option("display.max_colwidth", 120)

In [ ]:
import math
import numpy as np
from collections import Counter
from stki_utils import corpus_texts, preprocess, build_inverted_index, load_corpus

judul = dict(zip(load_corpus().doc_id, load_corpus().judul))
docs = {d: preprocess(t) for d, t in corpus_texts().items()}
index = build_inverted_index(docs)
N = len(docs)
vocab = list(index)
print(N, "dokumen,", len(vocab), "term")

30 dokumen, 472 term


## Langkah 1 — Term frequency dan variasinya

In [ ]:
tf_d10 = Counter(docs["D10"])
pd.DataFrame({
    "tf mentah": tf_d10,
    "tf log (1+log10 tf)": {t: 1 + math.log10(v) for t, v in tf_d10.items()},
}).sort_values("tf mentah", ascending=False).head(8)

,tf mentah,tf log (1+log10 tf)
tani,3,1.477121
cerdas,2,1.301030
sensor,2,1.301030
bas,1,1.000000
iot,1,1.000000
manfaat,1,1.000000
internet,1,1.000000
of,1,1.000000


## Langkah 2 — Document frequency dan IDF
Term yang muncul di banyak dokumen kurang mampu membedakan dokumen sehingga diberi bobot IDF kecil.

In [ ]:
idf = {t: math.log10(N / len(p)) for t, p in index.items()}
tabel_idf = pd.DataFrame({"df": {t: len(index[t]) for t in vocab}, "idf": idf})
pd.concat([tabel_idf.sort_values("df", ascending=False).head(5),
           tabel_idf.sort_values("df").head(5)]).round(3)

,df,idf
guna,11,0.436
lalu,9,0.523
bantu,8,0.574
tani,7,0.632
data,6,0.699
3m,1,1.477
acu,1,1.477
aedes,1,1.477
aegypti,1,1.477
ahli,1,1.477


## Langkah 3 — Matriks TF-IDF dokumen

In [ ]:
def tfidf_vector(tokens):
    return {t: (1 + math.log10(tf)) * idf[t] for t, tf in Counter(tokens).items() if t in idf}

M = pd.DataFrame(0.0, index=list(docs), columns=vocab)
for d, tokens in docs.items():
    for t, w in tfidf_vector(tokens).items():
        M.loc[d, t] = w
print("Ukuran matriks TF-IDF:", M.shape)
M.loc[["D07", "D08", "D09", "D10"], ["tani", "air", "pupuk", "sensor", "padi"]].round(3)

Ukuran matriks TF-IDF: (30, 472)


,tani,air,pupuk,sensor,padi
D07,0.632,0.875,0.000,0.00,2.182
D08,0.822,0.000,1.737,0.00,0.000
D09,0.632,0.875,0.000,0.00,0.000
D10,0.934,0.875,1.176,1.53,0.000


## Langkah 4 — Vektor query dan cosine similarity

In [ ]:
def cosine(u, v):
    nu, nv = np.linalg.norm(u), np.linalg.norm(v)
    return float(u @ v / (nu * nv)) if nu and nv else 0.0

query = "teknologi pertanian modern"
q_tokens = preprocess(query)
q_vec = pd.Series(0.0, index=vocab)
for t, w in tfidf_vector(q_tokens).items():
    q_vec[t] = w
print("Term query:", q_tokens, "| term di luar kosakata:", [t for t in q_tokens if t not in idf])

skor = {d: cosine(q_vec.values, M.loc[d].values) for d in docs}
hasil = sorted(skor.items(), key=lambda x: (-x[1], x[0]))[:5]
pd.DataFrame([(d, judul[d], round(s, 4)) for d, s in hasil], columns=["doc_id", "judul", "cosine"])

Term query: ['teknologi', 'tani', 'modern'] | term di luar kosakata: ['modern']


,doc_id,judul,cosine
0,D10,Pertanian Cerdas Berbasis IoT,0.2158
1,D06,Layanan Telemedisin,0.1442
2,D27,Literasi Digital untuk Pelajar,0.1421
3,D12,Harga Gabah Saat Panen Raya,0.0809
4,D08,Membuat Pupuk Organik,0.0670


## Langkah 5 — Implementasi efisien dengan inverted index
Menghitung cosine terhadap semua dokumen boros. Dengan inverted index, cukup dokumen yang memuat minimal satu term query yang diberi skor (*term-at-a-time*). Implementasi ini tersedia sebagai `TfidfModel` di `stki_utils.py`.

In [ ]:
from stki_utils import TfidfModel

model = TfidfModel(docs)

def cari(query, k=5):
    hasil = model.search(preprocess(query), k=k)
    return pd.DataFrame([(d, judul[d], round(s, 4)) for d, s in hasil], columns=["doc_id", "judul", "skor"])

cek = model.search(q_tokens, k=5)
print("Sama dengan perhitungan matriks:", [d for d, _ in cek] == [d for d, _ in hasil])
cari("pemasaran digital UMKM")

Sama dengan perhitungan matriks: True


,doc_id,judul,skor
0,D19,Digitalisasi UMKM,0.3999
1,D22,Kredit Usaha Rakyat,0.1612
2,D24,Investasi Saham bagi Pemula,0.1103
3,D27,Literasi Digital untuk Pelajar,0.0819
4,D21,Pembayaran Digital dengan QRIS,0.0790


In [ ]:
cari("pembelajaran daring mahasiswa")

,doc_id,judul,skor
0,D25,Pembelajaran Daring di Perguruan Tinggi,0.3535
1,D29,Perpustakaan Digital,0.1203
2,D28,Beasiswa Pendidikan Tinggi,0.1124
3,D26,Kurikulum Merdeka,0.0595
4,D14,Kecerdasan Buatan dalam Kehidupan,0.0587


## Tugas Step 4

1. Hitung sendiri bobot TF-IDF kata `tani` pada dokumen D10. Setelah itu, bandingkan hasil hitunganmu dengan hasil program.
2. Coba tiga cara menghitung TF: mentah, logaritmik, dan biner. Untuk query `harga hasil panen petani`, apakah urutan 5 dokumen teratas berubah? Jelaskan perbedaannya.
3. Query `pembelajaran daring mahasiswa` menampilkan dokumen D14 tentang Kecerdasan Buatan. Menurutmu, kenapa dokumen ini bisa muncul? Jelaskan kaitannya dengan preprocessing dan berikan saran untuk memperbaikinya.
4. (Opsional) Bandingkan hasil program dengan `TfidfVectorizer` dari scikit-learn. Pengaturan apa yang perlu diubah supaya hasilnya lebih mirip dengan perhitungan manual?

## Jawaban Tugas 1: Bobot TF-IDF term `tani` pada dokumen D10

**Perhitungan manual**

1. Setelah preprocessing, term `tani` muncul **3 kali** di D10: kata "Pertanian" 2 kali (pada judul dan kalimat pertama) dan kata "petani" 1 kali, yang ketiganya di-stem menjadi `tani`. Jadi $tf_{tani,D10} = 3$.
2. Term `tani` muncul di **7 dokumen** dari $N = 30$ dokumen (D07, D08, D09, D10, D11, D12, D23), sehingga $df_{tani} = 7$.
3. Hitung IDF:

$$idf_{tani} = \log_{10}\frac{N}{df_{tani}} = \log_{10}\frac{30}{7} = \log_{10}(4.2857) \approx 0.6320$$

4. Hitung TF logaritmik:

$$1 + \log_{10}(3) = 1 + 0.4771 = 1.4771$$

5. Kalikan keduanya:

$$w_{tani,D10} = 1.4771 \times 0.6320 \approx 0.9336$$

Sel berikut mencocokkan hasil manual dengan hasil program.

In [ ]:
# Cocokkan perhitungan manual dengan matriks TF-IDF (Langkah 3)
tf_tani = Counter(docs["D10"])["tani"]
df_tani = len(index["tani"])
manual = (1 + math.log10(tf_tani)) * math.log10(N / df_tani)

print(f"tf(tani, D10)  = {tf_tani}")
print(f"df(tani)       = {df_tani}  (dokumen: {sorted(index['tani'])})")
print(f"idf(tani)      = log10({N}/{df_tani}) = {math.log10(N / df_tani):.4f}")
print(f"1 + log10(tf)  = {1 + math.log10(tf_tani):.4f}")
print(f"bobot manual   = {manual:.4f}")
print(f"bobot program  = {M.loc['D10', 'tani']:.4f}   (matriks M, Langkah 3)")
print("Cocok:", np.isclose(manual, M.loc["D10", "tani"]))

# Catatan: vektor di TfidfModel sudah dinormalisasi (panjang vektor = 1),
# sehingga nilainya lebih kecil daripada bobot mentah di atas.
norma_d10 = np.linalg.norm(M.loc["D10"].values)
print(f"\nPanjang vektor D10 = {norma_d10:.4f}")
print(f"Bobot ternormalisasi manual  = {manual / norma_d10:.4f}")
print(f"Bobot di TfidfModel          = {model.doc_vectors['D10']['tani']:.4f}")

tf(tani, D10)  = 3
df(tani)       = 7  (dokumen: ['D07', 'D08', 'D09', 'D10', 'D11', 'D12', 'D23'])
idf(tani)      = log10(30/7) = 0.6320
1 + log10(tf)  = 1.4771
bobot manual   = 0.9336
bobot program  = 0.9336   (matriks M, Langkah 3)
Cocok: True

Panjang vektor D10 = 6.2286
Bobot ternormalisasi manual  = 0.1499
Bobot di TfidfModel          = 0.1499


## Jawaban Tugas 2: Perbandingan skema TF (mentah, logaritmik, biner)

Query `harga hasil panen petani` diproses menjadi term `harga`, `hasil`, `panen`, `tani`. Ketiga skema memakai IDF dan normalisasi cosine yang sama, hanya komponen TF yang berbeda:

| Skema | Rumus TF |
|---|---|
| Mentah | $tf$ |
| Logaritmik | $1 + \log_{10} tf$ |
| Biner | 1 jika term ada, 0 jika tidak |

In [ ]:
from stki_utils import load_qrels, average_precision

def normalisasi(v):
    n = math.sqrt(sum(x * x for x in v.values()))
    return {t: x / n for t, x in v.items()} if n else {}

def bobot_tf(tf, skema):
    return {"mentah": tf, "log": 1 + math.log10(tf), "biner": 1}[skema]

def peringkat(q_tokens, skema):
    """Ranked retrieval dengan skema TF tertentu (IDF dan cosine tetap sama)."""
    dv = {d: normalisasi({t: bobot_tf(c, skema) * idf[t] for t, c in Counter(tok).items()})
          for d, tok in docs.items()}
    qv = normalisasi({t: bobot_tf(c, skema) * idf[t] for t, c in Counter(q_tokens).items() if t in idf})
    skor = {d: sum(w * dv[d].get(t, 0.0) for t, w in qv.items()) for d in docs}
    return sorted([(d, s) for d, s in skor.items() if s > 0], key=lambda x: (-x[1], x[0]))

q7 = "harga hasil panen petani"
q7_tokens = preprocess(q7)
print("Term query:", q7_tokens)
print(pd.DataFrame({t: {"df": len(index[t]), "idf": round(idf[t], 3)} for t in q7_tokens}).T, "\n")

hasil_skema = {s: peringkat(q7_tokens, s) for s in ["mentah", "log", "biner"]}

# Tabel top-5 berdampingan: "doc_id (skor)"
tabel = pd.DataFrame({
    f"TF {s}": [f"{d} ({sk:.4f})" for d, sk in r[:5]] for s, r in hasil_skema.items()
}, index=[f"#{i}" for i in range(1, 6)])
display(tabel)

# Evaluasi dengan qrels Q7 (D12 bernilai 2, D07/D23/D20 bernilai 1)
relevan_q7 = set(load_qrels()["Q7"])
print("Dokumen relevan Q7:", sorted(relevan_q7))
pd.DataFrame({
    f"TF {s}": {"P@5": sum(d in relevan_q7 for d, _ in r[:5]) / 5,
                "AP": round(average_precision([d for d, _ in r], relevan_q7), 4)}
    for s, r in hasil_skema.items()
}).T

Term query: ['harga', 'hasil', 'panen', 'tani']
        df    idf
harga  3.0  1.000
hasil  3.0  1.000
panen  2.0  1.176
tani   7.0  0.632 



,TF mentah,TF log,TF biner
#1,D12 (0.4208),D12 (0.3224),D12 (0.2578)
#2,D07 (0.1802),D07 (0.2265),D07 (0.2423)
#3,D20 (0.1185),D23 (0.1098),D23 (0.1167)
#4,D10 (0.0899),D20 (0.1075),D20 (0.0894)
#5,D23 (0.0818),D18 (0.0827),D18 (0.0885)


Dokumen relevan Q7: ['D07', 'D12', 'D20', 'D23']


,P@5,AP
TF mentah,0.8,0.95
TF log,0.8,1.00
TF biner,0.8,1.00


**Analisis perubahan urutan**

| Posisi | TF mentah | TF logaritmik | TF biner |
|---|---|---|---|
| 1 | D12 | D12 | D12 |
| 2 | D07 | D07 | D07 |
| 3 | D20 | D23 | D23 |
| 4 | D10 | D20 | D20 |
| 5 | D23 | D18 | D18 |

1. **D12 dan D07 selalu di posisi 1 dan 2.** Keduanya memuat beberapa term query sekaligus, termasuk `panen` yang memiliki IDF tertinggi (1.176), sehingga posisinya aman pada ketiga skema.
2. **TF mentah memberi pengaruh terlalu besar pada pengulangan kata.** Di D12, `harga` muncul 3 kali dan `tani` 3 kali, sehingga skornya naik menjadi 0.4208 (dengan TF log hanya 0.3224). D10 (Pertanian Cerdas Berbasis IoT) naik ke posisi 4 semata-mata karena `tani` muncul 3 kali, padahal dokumen itu tidak membahas harga maupun panen dan tidak ada di qrels Q7. Akibatnya D23 yang relevan turun ke posisi 5.
3. **TF logaritmik dan biner menghasilkan urutan top-5 yang sama**, hanya skornya yang berbeda. Pada skema biner, selisih D12 dan D07 mengecil (0.2578 vs 0.2423) karena pengulangan kata di D12 tidak lagi dihargai.
4. **Evaluasi dengan qrels Q7:** P@5 ketiganya sama (0.8) karena masing-masing memuat 4 dokumen relevan di lima teratas. Perbedaan terlihat pada AP: TF mentah 0.95, sedangkan TF logaritmik dan biner 1.0 karena keempat dokumen relevan berada di posisi 1 sampai 4.

**Kesimpulan:** TF logaritmik menjadi pilihan yang seimbang. Frekuensi kata tetap dihitung, tetapi pengaruh pengulangan diredam. TF mentah rentan mengangkat dokumen yang hanya mengulang satu term umum, sedangkan TF biner membuang seluruh informasi frekuensi (masih aman di korpus ini karena dokumennya pendek).

## Jawaban Tugas 3: Mengapa `pembelajaran daring mahasiswa` mengembalikan D14?

D14 (Kecerdasan Buatan dalam Kehidupan) membahas *machine learning*, topik yang berbeda dari kuliah daring. Sel berikut menelusuri asal kecocokannya.

In [ ]:
import re
from collections import defaultdict
from stki_utils import tokenize, stem

q8 = "pembelajaran daring mahasiswa"
q8_tokens = preprocess(q8)
print("Query setelah preprocessing:", q8_tokens)
print("Hasil baseline (top-8):")
display(pd.DataFrame([(d, judul[d], round(s, 4)) for d, s in model.search(q8_tokens, k=8)],
                     columns=["doc_id", "judul", "skor"]))

# 1) Term query mana yang cocok di D14?
print("Kemunculan term query di D14:", {t: Counter(docs["D14"])[t] for t in q8_tokens})

# 2) Kata asli apa saja di korpus yang di-stem menjadi 'ajar'?
rows = []
for d, teks in corpus_texts().items():
    asli = [w for w in tokenize(teks) if stem(w) == "ajar"]
    if asli:
        rows.append((d, judul[d], Counter(asli)))
pd.DataFrame(rows, columns=["doc_id", "judul", "kata asli yang menjadi 'ajar'"])

Query setelah preprocessing: ['ajar', 'daring', 'mahasiswa']
Hasil baseline (top-8):


,doc_id,judul,skor
0,D25,Pembelajaran Daring di Perguruan Tinggi,0.3535
1,D29,Perpustakaan Digital,0.1203
2,D28,Beasiswa Pendidikan Tinggi,0.1124
3,D26,Kurikulum Merdeka,0.0595
4,D14,Kecerdasan Buatan dalam Kehidupan,0.0587
5,D17,Pemrosesan Bahasa Alami Bahasa Indonesia,0.0510
6,D27,Literasi Digital untuk Pelajar,0.0455
7,D30,Gizi Anak Sekolah,0.0414


Kemunculan term query di D14: {'ajar': 2, 'daring': 0, 'mahasiswa': 0}


,doc_id,judul,kata asli yang menjadi 'ajar'
0,D14,Kecerdasan Buatan dalam Kehidupan,"{'belajar': 1, 'pembelajaran': 1}"
1,D17,Pemrosesan Bahasa Alami Bahasa Indonesia,"{'pembelajaran': 1, 'ajar': 1}"
2,D25,Pembelajaran Daring di Perguruan Tinggi,{'pembelajaran': 3}
3,D26,Kurikulum Merdeka,{'pembelajaran': 2}
4,D27,Literasi Digital untuk Pelajar,{'pelajar': 1}
5,D30,Gizi Anak Sekolah,{'belajar': 1}


**Penyebab**

1. **Stemming menyatukan kata yang konteksnya berbeda.** Sastrawi mengubah `pembelajaran`, `belajar`, `ajar`, dan `pelajar` menjadi akar yang sama, yaitu `ajar`. Di D14 terdapat "komputer **belajar** dari data" dan "**Pembelajaran** mesin", sehingga `ajar` dihitung dua kali (tf = 2) untuk makna *machine learning*.
2. **Model bag-of-words membuang konteks dan urutan kata.** Frasa `pembelajaran daring` diperlakukan sebagai dua term yang berdiri sendiri, sehingga dokumen yang hanya memuat salah satunya sudah mendapat skor.
3. **Tidak ada syarat jumlah term yang cocok.** D14 hanya cocok pada 1 dari 3 term (`ajar`), sedangkan `daring` dan `mahasiswa` tidak ada. Skornya kecil (0.0587 dibanding 0.3535 pada D25), tetapi tetap masuk top-5 karena korpus hanya berisi 30 dokumen dan hanya 8 dokumen yang memiliki skor lebih dari 0.

Sel berikut menguji beberapa usulan perbaikan. Fungsi pembantu ditulis di notebook ini, `stki_utils.py` tidak diubah supaya modul lain tidak terpengaruh.

In [ ]:
# --- Usulan perbaikan -------------------------------------------------------
# A) Tanpa stemming
docs_ns = {d: preprocess(t, use_stemming=False) for d, t in corpus_texts().items()}
model_ns = TfidfModel(docs_ns)
hasil_A = model_ns.search(preprocess(q8, use_stemming=False), k=5)

# B) Pencocokan frasa: tambahkan bigram (mis. 'ajar_daring') sebagai term tambahan
def dengan_bigram(tokens):
    return tokens + [a + "_" + b for a, b in zip(tokens, tokens[1:])]

model_bg = TfidfModel({d: dengan_bigram(t) for d, t in docs.items()})
hasil_B = model_bg.search(dengan_bigram(q8_tokens), k=5)

# C) Syarat minimal term yang cocok (minimum should match)
def cari_min_match(model, q_tokens, min_match=2, k=5):
    qv = model.query_vector(q_tokens)
    skor, cocok = defaultdict(float), Counter()
    for t, wq in qv.items():
        for d in model.index[t]:
            skor[d] += wq * model.doc_vectors[d][t]
            cocok[d] += 1
    hasil = [(d, s) for d, s in skor.items() if cocok[d] >= min_match]
    return sorted(hasil, key=lambda x: (-x[1], x[0]))[:k]

hasil_C = cari_min_match(model, q8_tokens, min_match=2)

relevan_q8 = set(load_qrels()["Q8"])      # D25, D28, D26
varian = {
    "Baseline": model.search(q8_tokens, k=5),
    "A. Tanpa stemming": hasil_A,
    "B. Unigram + bigram": hasil_B,
    "C. Minimal 2 term cocok": hasil_C,
}
baris = []
for nama, hsl in varian.items():
    ranked = [d for d, _ in hsl]
    top5 = ranked[:5]
    baris.append({
        "varian": nama,
        "top-5": ", ".join(f"{d} ({s:.3f})" for d, s in hsl),
        "D14 muncul?": "ya" if "D14" in ranked else "tidak",
        "presisi": round(sum(d in relevan_q8 for d in top5) / len(top5), 2),
        "recall": round(sum(d in relevan_q8 for d in top5) / len(relevan_q8), 2),
    })
pd.DataFrame(baris).set_index("varian")

,top-5,D14 muncul?,presisi,recall
varian,,,,
Baseline,"D25 (0.354), D29 (0.120), D28 (0.112), D26 (0.060), D14 (0.059)",ya,0.6,1.00
A. Tanpa stemming,"D25 (0.362), D29 (0.112), D28 (0.103), D26 (0.085), D14 (0.061)",ya,0.6,1.00
B. Unigram + bigram,"D25 (0.297), D29 (0.060), D28 (0.057), D26 (0.029), D14 (0.028)",ya,0.6,1.00
C. Minimal 2 term cocok,D25 (0.354),tidak,1.0,0.33


**Hasil dan usulan perbaikan**

| Varian | Efek pada D14 | Catatan |
|---|---|---|
| A. Tanpa stemming | Tetap muncul (skor 0.0610) | D14 secara harfiah memuat kata `pembelajaran` ("pembelajaran mesin"), sehingga menghapus stemming tidak menyelesaikan masalah. |
| B. Unigram + bigram | Tetap muncul, skornya turun menjadi 0.0281 | Frasa `ajar_daring` hanya ada di D25, jadi dokumen yang memuat frasa persis mendapat keunggulan lebih jelas. Urutan top-5 tidak berubah. |
| C. Minimal 2 term cocok | Hilang | Hanya D25 yang lolos sehingga presisi naik, tetapi recall turun karena D28 dan D26 (relevan, hanya cocok 1 term) ikut tersaring. |

Rekomendasi:

1. Gunakan **unigram + bigram** agar dokumen yang memuat frasa persis (`pembelajaran daring`) mendapat skor lebih tinggi.
2. Terapkan **minimum should match** atau ambang skor hanya bila prioritasnya presisi, dengan konsekuensi recall turun.
3. Perlu dicatat bahwa D14 dan D26 secara leksikal hampir tidak bisa dibedakan (sama-sama hanya cocok pada `ajar`, skor 0.0587 dan 0.0595), padahal D26 relevan dan D14 tidak. Pemisahan yang andal membutuhkan pemahaman makna, yaitu pendekatan semantik pada Step 9 (Word2Vec) dan Step 10 (BERT).

## Jawaban Tugas 4 (Opsional): Perbandingan dengan `TfidfVectorizer` scikit-learn

Parameter yang perlu diatur agar mendekati implementasi manual:

| Parameter | Nilai | Alasan |
|---|---|---|
| `analyzer` | `lambda x: x` | Input sudah berupa list token hasil `preprocess()`, jadi sklearn tidak perlu tokenisasi, lowercase, atau stopword sendiri. |
| `sublinear_tf` | `True` | TF menjadi $1 + \ln tf$ (setara skema logaritmik). |
| `smooth_idf` | `False` | Menghilangkan penghalusan $(1+N)/(1+df)$ pada IDF. |
| `norm` | `"l2"` (default) | Normalisasi panjang vektor sehingga perkalian titik sama dengan cosine. |

Dua perbedaan tetap tersisa karena tidak bisa diatur lewat parameter: (1) IDF sklearn adalah $\ln(N/df) + 1$, ada tambahan **+1** yang tidak ada di rumus manual $\log_{10}(N/df)$, dan (2) `sublinear_tf` memakai logaritma natural, sedangkan manual memakai $\log_{10}$.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import normalize
from scipy import sparse
from stki_utils import load_queries, load_qrels, average_precision

ids = list(docs)
corpus_tokens = [docs[d] for d in ids]
queries = load_queries()
qrels_all = load_qrels()

def buat_sklearn(**kw):
    vec = TfidfVectorizer(analyzer=lambda x: x, **kw)
    return vec, vec.fit_transform(corpus_tokens)

def cari_sklearn(vec, X, q_tokens, k=30):
    skor = (X @ vec.transform([q_tokens]).T).toarray().ravel()
    hasil = [(ids[i], s) for i, s in enumerate(skor) if s > 0]
    return sorted(hasil, key=lambda x: (-x[1], x[0]))[:k]

# Pembanding eksak: blok bangunan sklearn + rumus manual (TF log10, IDF log10(N/df))
cv = CountVectorizer(analyzer=lambda x: x)
C = cv.fit_transform(corpus_tokens).tocsr().astype(float)
df_vec = np.asarray((C > 0).sum(axis=0)).ravel()
idf_manual = np.log10(len(ids) / df_vec)
C.data = 1 + np.log10(C.data)
X_eksak = normalize(C @ sparse.diags(idf_manual), norm="l2")

def cari_eksak(q_tokens, k=30):
    v = np.zeros(len(cv.vocabulary_))
    for t, f in Counter(t for t in q_tokens if t in cv.vocabulary_).items():
        v[cv.vocabulary_[t]] = (1 + math.log10(f)) * idf_manual[cv.vocabulary_[t]]
    n = np.linalg.norm(v)
    skor = X_eksak @ (v / n if n else v)
    hasil = [(ids[i], s) for i, s in enumerate(skor) if s > 0]
    return sorted(hasil, key=lambda x: (-x[1], x[0]))[:k]

konfigurasi = {
    "sklearn default": lambda qt: cari_sklearn(*vec_default, qt),
    "sklearn sublinear_tf + smooth_idf=False": lambda qt: cari_sklearn(*vec_tuned, qt),
    "sklearn + log10 + idf manual (eksak)": cari_eksak,
}
vec_default = buat_sklearn()
vec_tuned = buat_sklearn(sublinear_tf=True, smooth_idf=False)

baris = []
for nama, fungsi in konfigurasi.items():
    sama, selisih, ap = 0, 0.0, []
    for q, teks in queries.items():
        qt = preprocess(teks)
        manual_hasil = model.search(qt, k=30)
        lain = fungsi(qt)
        sama += [d for d, _ in manual_hasil[:5]] == [d for d, _ in lain[:5]]
        md_ = dict(manual_hasil)
        selisih = max([selisih] + [abs(s - md_.get(d, 0.0)) for d, s in lain])
        ap.append(average_precision([d for d, _ in lain], set(qrels_all[q])))
    baris.append({"konfigurasi": nama, "top-5 identik": f"{sama}/{len(queries)}",
                  "selisih skor maks": f"{selisih:.2e}", "MAP": round(float(np.mean(ap)), 4)})

map_manual = np.mean([average_precision([d for d, _ in model.search(preprocess(t), k=30)], set(qrels_all[q]))
                      for q, t in queries.items()])
print("MAP implementasi manual (TfidfModel):", round(float(map_manual), 4))
pd.DataFrame(baris).set_index("konfigurasi")

MAP implementasi manual (TfidfModel): 0.762


,top-5 identik,selisih skor maks,MAP
konfigurasi,,,
sklearn default,3/8,2.04e-01,0.6918
sklearn sublinear_tf + smooth_idf=False,3/8,9.87e-02,0.7451
sklearn + log10 + idf manual (eksak),8/8,1.67e-16,0.7620


**Kesimpulan Tugas 4**

- Dengan `analyzer=lambda x: x`, `sublinear_tf=True`, dan `smooth_idf=False`, skor sklearn jauh lebih dekat ke implementasi manual dibanding konfigurasi default (selisih skor maksimum turun dari sekitar 0.20 menjadi sekitar 0.10), dan MAP naik dari 0.692 menjadi 0.745 (manual: 0.762).
- Urutan top-5 belum identik untuk semua query karena dua perbedaan yang tersisa, yaitu tambahan **+1** pada IDF sklearn dan basis logaritma pada TF ($\ln$ pada sklearn, $\log_{10}$ pada manual). Pada korpus kecil, skor antar dokumen di posisi bawah sangat tipis sehingga selisih kecil ini cukup untuk menukar urutannya.
- Hasil yang **identik** (selisih skor sekitar $10^{-16}$) baru didapat bila TF $1+\log_{10} tf$ dan IDF $\log_{10}(N/df)$ dihitung sendiri lalu dinormalisasi dengan `normalize(..., norm="l2")`, seperti pada pembanding eksak di atas. Artinya `TfidfVectorizer` cocok untuk praktik, sedangkan rumus modul ini tidak bisa direplikasi persis hanya lewat parameternya.